<a href="https://colab.research.google.com/github/noor00-ai/fly_rank_intern/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noor00-ai/fly_rank_intern/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I selected Random Forest Classifier for this ranking signal analysis task.

The dataset contains multiple ranking signals such as CTR, impressions, position, freshness, and engagement metrics. Random Forest can capture non-linear relationships between these signals and identify which features contribute most to content actions.

This model is suitable as a baseline because it is interpretable through feature importance and can be compared fairly against the Week-4 rule-based baseline.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Load dataset
df = pd.read_csv("/content/content_refresh_anonymized.csv")

df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


The dataset does not contain a future outcome label.
To avoid leakage, the target is created only from existing ranking signals.

A content item is marked as needing action when it has:
- declining trend
- low CTR
- poor average position

This represents a decision-support baseline, not a final business label.

In [12]:
df["target"] = (
    ((df["trend_direction"] == "down") &
     (df["ctr"] < df["ctr"].median()))
    |
    (df["avg_position"] > 20)
).astype(int)


df["target"].value_counts()

,count
target,
0,16482
1,13518


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I used a grouped split based on client_id.

Grouping by client prevents the same client appearing in both training and testing data. This gives a more realistic evaluation because the model is tested on unseen client groups instead of memorizing client-specific patterns.

In [13]:
from sklearn.model_selection import GroupShuffleSplit

features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "trend_pct"
]


model_data = df[features + ["client_id", "target"]].dropna()


X = model_data[features]
y = model_data["target"]
groups = model_data["client_id"]


splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)


train_idx, test_idx = next(
    splitter.split(X, y, groups)
)


X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]


print("Training size:", X_train.shape)
print("Testing size:", X_test.shape)

Training size: (12493, 13)
Testing size: (5424, 13)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The Random Forest model is trained using the same data split.
Performance is compared using F1-score because the task focuses on identifying useful content actions.

In [14]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)


model.fit(X_train, y_train)


predictions = model.predict(X_test)


model_results = pd.DataFrame({
    "Model": ["Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, predictions)
    ],
    "Precision": [
        precision_score(y_test, predictions)
    ],
    "Recall": [
        recall_score(y_test, predictions)
    ],
    "F1 Score": [
        f1_score(y_test, predictions)
    ]
})


model_results

,Model,Accuracy,Precision,Recall,F1 Score
0,Random Forest,1.0,1.0,1.0,1.0


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The model was analyzed by checking incorrect predictions.

The errors show where the model disagrees with the rule-based baseline. These cases help identify weak signals and possible improvements.

Feature importance is also checked to understand what signals influence decisions.

In [15]:
error_analysis = X_test.copy()

error_analysis["actual"] = y_test.values
error_analysis["predicted"] = predictions


wrong_predictions = error_analysis[
    error_analysis["actual"] != error_analysis["predicted"]
]


wrong_predictions.head(10)

,search_volume,competition,cpc,word_count,impressions_90d,clicks_90d,sessions_90d,days_since_last_update,ctr,avg_position,engagement_rate,scroll_rate,trend_pct,actual,predicted


In [16]:
importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
})


importance.sort_values(
    by="importance",
    ascending=False
)

,feature,importance
9,avg_position,0.463406
8,ctr,0.192709
5,clicks_90d,0.123099
12,trend_pct,0.104694
4,impressions_90d,0.041984
6,sessions_90d,0.022416
3,word_count,0.017888
7,days_since_last_update,0.011519
10,engagement_rate,0.008055
11,scroll_rate,0.006164


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.